# Praktikum Klasifikasi: Biner, Multikelas, dan Multilabel

Notebook ini berisi **9 dataset Kaggle** (3 per jenis klasifikasi) yang jarang dipakai di tutorial umum.
Tiap dataset diunduh langsung dari Kaggle, **diidentifikasi** jenis klasifikasinya, lalu dilatih dan dievaluasi dengan model baseline.

| No | Jenis | Dataset | Ukuran | Identifikasi target | Link |
|---|---|---|---|---|---|
| A1 | Biner | Exoplanet Hunting (Kepler) | 5.087 bintang (train) | 1 kolom, 2 nilai (tanpa/punya exoplanet), sangat tidak seimbang | https://www.kaggle.com/datasets/keplersmachines/kepler-labelled-time-series-data |
| A2 | Biner | Pistachio | 2.148 data | 1 kolom, 2 jenis (Kirmizi / Siirt) | https://www.kaggle.com/datasets/muratkokludataset/pistachio-dataset |
| A3 | Biner | Pokemon (Legendary) | 802 Pokemon | `is_legendary` 0/1 | https://www.kaggle.com/datasets/rounakbanik/pokemon |
| B1 | Multikelas | Date Fruit | 898 data, 34 fitur | 1 kolom, 7 jenis kurma | https://www.kaggle.com/datasets/muratkokludataset/date-fruit-datasets |
| B2 | Multikelas | Stellar Classification SDSS17 | 100.000 observasi | 1 kolom, 3 kelas (GALAXY, STAR, QSO) | https://www.kaggle.com/datasets/fedesoriano/stellar-classification-dataset-sdss17 |
| B3 | Multikelas | Student Dropout & Academic Success | 4.424 mahasiswa | 1 kolom, 3 kelas (Dropout, Enrolled, Graduate) | https://www.kaggle.com/datasets/thedevastator/higher-education-predictors-of-student-retention |
| C1 | Multilabel | Pokemon (tipe) | 898 Pokemon | Type 1 + Type 2 menjadi 18 label biner | https://www.kaggle.com/datasets/jaidalmotra/pokemon-dataset |
| C2 | Multilabel | NIH Chest X-ray (metadata) | 112.120 gambar | 14 label penyakit, satu gambar bisa punya beberapa | https://www.kaggle.com/datasets/nih-chest-xrays/data |
| C3 | Multilabel | Indonesian Abusive & Hate Speech Twitter | 13.169 tweet | 12 kolom label biner yang bisa aktif bersamaan | https://www.kaggle.com/datasets/ilhamfp31/indonesian-abusive-and-hate-speech-twitter-text |

## Cara membedakan ketiga jenis klasifikasi

| Jenis | Bentuk target | Contoh |
|---|---|---|
| Biner | 1 kolom, tepat 2 nilai | M/B, 0/1 |
| Multikelas | 1 kolom, lebih dari 2 nilai, tiap baris tepat satu kelas | 3 kelas bintang/galaksi/kuasar |
| Multilabel | Beberapa kolom biner (satu per label), bisa aktif bersamaan | Pokemon Fire + Flying |

Ciri multilabel: **label cardinality** (rata-rata jumlah label per baris) bisa lebih dari 1.

> **Catatan penting**
> - Notebook ini mengunduh data lewat `kagglehub` (butuh internet). Dataset publik biasanya bisa diunduh tanpa login. Jika diminta login, jalankan `kagglehub.login()` atau set `KAGGLE_USERNAME` dan `KAGGLE_KEY`.
> - Nama file/kolom diambil otomatis. Jika ada yang berbeda dari perkiraan, notebook akan menampilkan daftar kolom yang tersedia, lalu sesuaikan nama kolomnya di sel terkait.
> - Skor model hanyalah **baseline**. Tujuan utamanya adalah memahami cara mengidentifikasi dan mengevaluasi tiap jenis klasifikasi.

## 0. Setup

In [ ]:
!pip -q install kagglehub openpyxl xlrd

In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MultiLabelBinarizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, classification_report,
    ConfusionMatrixDisplay, hamming_loss,
)

try:
    from IPython.display import display
except ImportError:  # kalau dijalankan di luar Jupyter
    display = print

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42
RESULTS = []  # semua hasil evaluasi dikumpulkan di sini untuk ringkasan akhir

In [ ]:
# ---------- Helper: unduh & baca data ----------
def download(handle, path=None):
    '''Unduh dataset Kaggle publik lewat kagglehub. Return Path folder (atau file jika path diisi).'''
    p = kagglehub.dataset_download(handle, path=path) if path else kagglehub.dataset_download(handle)
    return Path(p)


def find_file(folder, patterns=(), exts=(".csv", ".xlsx", ".xls")):
    '''Cari file tabular di folder (rekursif). Prioritaskan nama yang mengandung salah satu pola.'''
    files = sorted(p for p in Path(folder).rglob("*") if p.suffix.lower() in exts)
    if not files:
        raise FileNotFoundError(f"Tidak ada file tabular (csv/xlsx/xls) di {folder}")
    for pat in patterns:
        hits = [p for p in files if pat.lower() in p.name.lower()]
        if hits:
            return hits[0]
    return files[0]


def read_any(path, **kw):
    path = Path(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path, **kw)
    return pd.read_excel(path, **kw)


def pick_col(df, candidates):
    '''Ambil nama kolom asli dari daftar kandidat (tidak peka huruf besar/kecil).'''
    lower = {str(c).lower().strip(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    raise KeyError(f"Kolom {candidates} tidak ditemukan. Kolom tersedia: {list(df.columns)[:40]}")


def clean_features(X):
    X = X.copy()
    for c in X.select_dtypes(include="bool").columns:
        X[c] = X[c].astype(int)
    return X.replace([np.inf, -np.inf], np.nan)


def to_binary(y):
    '''Ubah target 2 kelas (teks/angka) menjadi 0/1 dan tampilkan pemetaannya.'''
    y = pd.Series(y)
    codes, uniques = pd.factorize(y.astype(str).str.strip(), sort=True)
    assert len(uniques) == 2, f"Target bukan biner, ada {len(uniques)} kelas: {list(uniques)}"
    print("Pemetaan kelas:", {u: i for i, u in enumerate(uniques)})
    return pd.Series(codes, index=y.index), list(uniques)


def make_preprocessor(X, scale=True):
    num_cols = X.select_dtypes(include="number").columns.tolist()
    cat_cols = [c for c in X.columns if c not in num_cols]
    parts = []
    if num_cols:
        steps = [("imp", SimpleImputer(strategy="median"))]
        if scale:
            steps.append(("sc", StandardScaler()))
        parts.append(("num", Pipeline(steps), num_cols))
    if cat_cols:
        parts.append(("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols))
    return ColumnTransformer(parts)


def plot_dist(y, title, rot=0):
    ax = pd.Series(y).value_counts().sort_index().plot(kind="bar", figsize=(6, 3), color="#4C78A8")
    ax.set_title(title)
    ax.set_ylabel("jumlah")
    plt.xticks(rotation=rot)
    plt.tight_layout()
    plt.show()

In [ ]:
# ---------- Helper: evaluasi klasifikasi biner ----------
def run_binary(name, X, y, test_size=0.2):
    X = clean_features(X)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=test_size, stratify=y, random_state=RANDOM_STATE)
    print(f"Train: {Xtr.shape}, Test: {Xte.shape}, positif di test: {int(yte.sum())}")

    models = {
        "LogReg": Pipeline([("prep", make_preprocessor(Xtr)),
                            ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))]),
        "RandomForest": Pipeline([("prep", make_preprocessor(Xtr, scale=False)),
                                  ("clf", RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                                                 n_jobs=-1, random_state=RANDOM_STATE))]),
    }
    rows, fitted = [], {}
    for mname, model in models.items():
        model.fit(Xtr, ytr)
        pred = model.predict(Xte)
        proba = model.predict_proba(Xte)[:, 1]
        row = dict(dataset=name, task="biner", model=mname,
                   accuracy=accuracy_score(yte, pred),
                   precision=precision_score(yte, pred, zero_division=0),
                   recall=recall_score(yte, pred, zero_division=0),
                   f1=f1_score(yte, pred, zero_division=0),
                   roc_auc=roc_auc_score(yte, proba),
                   pr_auc=average_precision_score(yte, proba))
        rows.append(row)
        fitted[mname] = pred
    RESULTS.extend(rows)
    res = pd.DataFrame(rows).set_index("model").drop(columns=["dataset", "task"]).round(3)
    display(res)

    best = res["f1"].idxmax()
    print(f"Model terbaik (F1): {best}")
    ConfusionMatrixDisplay.from_predictions(yte, fitted[best], cmap="Blues")
    plt.title(f"{name} - {best}")
    plt.show()


# ---------- Helper: evaluasi klasifikasi multikelas ----------
def run_multiclass(name, X, y, test_size=0.2):
    X = clean_features(X)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=test_size, stratify=y, random_state=RANDOM_STATE)
    print(f"Train: {Xtr.shape}, Test: {Xte.shape}, jumlah kelas: {y.nunique()}")

    models = {
        "LogReg": Pipeline([("prep", make_preprocessor(Xtr)),
                            ("clf", LogisticRegression(max_iter=3000, class_weight="balanced"))]),
        "RandomForest": Pipeline([("prep", make_preprocessor(Xtr, scale=False)),
                                  ("clf", RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                                                 n_jobs=-1, random_state=RANDOM_STATE))]),
    }
    rows, fitted = [], {}
    for mname, model in models.items():
        model.fit(Xtr, ytr)
        pred = model.predict(Xte)
        rows.append(dict(dataset=name, task="multikelas", model=mname,
                         accuracy=accuracy_score(yte, pred),
                         f1_macro=f1_score(yte, pred, average="macro", zero_division=0),
                         f1_weighted=f1_score(yte, pred, average="weighted", zero_division=0)))
        fitted[mname] = pred
    RESULTS.extend(rows)
    res = pd.DataFrame(rows).set_index("model").drop(columns=["dataset", "task"]).round(3)
    display(res)

    best = res["f1_macro"].idxmax()
    print(f"Model terbaik (F1 macro): {best}")
    print(classification_report(yte, fitted[best], zero_division=0))
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(yte, fitted[best], cmap="Blues", ax=ax, xticks_rotation=45)
    ax.set_title(f"{name} - {best}")
    plt.tight_layout()
    plt.show()


# ---------- Helper: identifikasi & evaluasi klasifikasi multilabel ----------
def check_multilabel(Y, title):
    Y = pd.DataFrame(Y)
    per_row = Y.sum(axis=1)
    print(f"Jumlah label            : {Y.shape[1]}")
    print(f"Label cardinality       : {per_row.mean():.3f}  (rata-rata label per baris)")
    print(f"Label density           : {per_row.mean() / Y.shape[1]:.3f}")
    print(f"Baris dengan >1 label   : {(per_row > 1).mean():.1%}")
    print(f"Baris tanpa label       : {(per_row == 0).mean():.1%}")
    print("=> Multilabel" if (per_row > 1).any() else "=> Bukan multilabel (tidak ada baris dengan >1 label)")
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
    Y.sum().sort_values(ascending=False).plot(kind="bar", ax=axes[0], color="#4C78A8")
    axes[0].set_title(f"{title}: frekuensi tiap label")
    per_row.value_counts().sort_index().plot(kind="bar", ax=axes[1], color="#F58518")
    axes[1].set_title("Jumlah label per baris")
    plt.tight_layout()
    plt.show()


def run_multilabel(name, models, Xtr, Xte, Ytr, Yte, label_names):
    Ytr, Yte = np.asarray(Ytr), np.asarray(Yte)
    rows, preds = [], {}
    for mname, model in models.items():
        model.fit(Xtr, Ytr)
        pred = np.asarray(model.predict(Xte))
        preds[mname] = pred
        rows.append(dict(dataset=name, task="multilabel", model=mname,
                         hamming_loss=hamming_loss(Yte, pred),
                         subset_acc=accuracy_score(Yte, pred),
                         f1_micro=f1_score(Yte, pred, average="micro", zero_division=0),
                         f1_macro=f1_score(Yte, pred, average="macro", zero_division=0),
                         f1_samples=f1_score(Yte, pred, average="samples", zero_division=0)))
    RESULTS.extend(rows)
    res = pd.DataFrame(rows).set_index("model").drop(columns=["dataset", "task"]).round(3)
    display(res)

    best = res["f1_micro"].idxmax()
    per_label = f1_score(Yte, preds[best], average=None, zero_division=0)
    pd.Series(per_label, index=label_names).sort_values().plot(kind="barh", figsize=(6, max(3, 0.3 * len(label_names))),
                                                                 color="#54A24B")
    plt.title(f"{name} - F1 per label ({best})")
    plt.tight_layout()
    plt.show()

---
# A. Klasifikasi Biner
Target hanya punya **2 kelas**. Karena data sering tidak seimbang, selain accuracy kita lihat precision, recall, F1, ROC-AUC, dan PR-AUC.

## A1. Exoplanet Hunting in Deep Space (Kepler)
- **Data:** deret waktu intensitas cahaya (flux) ribuan bintang. Kolom `LABEL`: 1 = tanpa exoplanet, 2 = punya exoplanet.
- **Identifikasi biner:** satu kolom target dengan 2 nilai.
- **Tantangan:** hanya 37 bintang berlabel exoplanet pada data latih (sekitar 0,7%), jadi accuracy menyesatkan. Perhatikan recall dan PR-AUC.
- Link: https://www.kaggle.com/datasets/keplersmachines/kepler-labelled-time-series-data

In [ ]:
folder = download("keplersmachines/kepler-labelled-time-series-data")
df = read_any(find_file(folder, patterns=["exoTrain"]))
print(df.shape)

label_col = pick_col(df, ["LABEL"])
y = (df[label_col] == 2).astype(int)   # 1 = punya exoplanet
X = df.drop(columns=[label_col])

# --- Identifikasi ---
print("Nilai unik target:", sorted(df[label_col].unique()), "=> biner:", y.nunique() == 2)
print(y.value_counts().rename({0: "tanpa exoplanet", 1: "punya exoplanet"}))
plot_dist(y, "Exoplanet: distribusi kelas")

In [ ]:
run_binary("Exoplanet Kepler", X, y)

## A2. Pistachio (Kirmizi vs Siirt)
- **Data:** fitur hasil olah citra dua jenis pistachio (2.148 sampel: 1.232 Kirmizi dan 916 Siirt).
- **Identifikasi biner:** kolom `Class` hanya berisi 2 jenis.
- Link: https://www.kaggle.com/datasets/muratkokludataset/pistachio-dataset

In [ ]:
folder = download("muratkokludataset/pistachio-dataset")
path = find_file(folder, patterns=["16_Features", "16 Features", "Pistachio_16", "Pistachio"])
print("File:", path.name)
df = read_any(path)
print(df.shape)

target = pick_col(df, ["Class", "Target", "Label"])
y, classes = to_binary(df[target])
X = df.drop(columns=[target])

# --- Identifikasi ---
print(df[target].value_counts())
plot_dist(df[target], "Pistachio: distribusi kelas")
display(df.head())

In [ ]:
run_binary("Pistachio", X, y)

## A3. Pokemon: Legendary atau bukan
- **Data:** 802 Pokemon dari 7 generasi (base stats, tinggi, berat, rasio gender, dan lain-lain). Kolom `is_legendary` bernilai 1/0.
- **Identifikasi biner:** target 0/1. Sekitar 8% saja yang legendary, jadi tidak seimbang.
- Link: https://www.kaggle.com/datasets/rounakbanik/pokemon

In [ ]:
folder = download("rounakbanik/pokemon")
df = read_any(find_file(folder, patterns=["pokemon"]))
print(df.shape)

target = pick_col(df, ["is_legendary", "legendary"])
y = df[target].astype(int)

drop_cols = [c for c in ["name", "japanese_name", "pokedex_number", "abilities", "classfication", "classification", target]
             if c in df.columns]
X = df.drop(columns=drop_cols)
if "capture_rate" in X.columns:   # kadang berisi teks, ubah ke angka
    X["capture_rate"] = pd.to_numeric(X["capture_rate"], errors="coerce")

# --- Identifikasi ---
print("Nilai unik target:", sorted(y.unique()), "=> biner:", y.nunique() == 2)
print(y.value_counts(normalize=True).round(3))
plot_dist(y, "Pokemon legendary: distribusi kelas")

In [ ]:
run_binary("Pokemon legendary", X, y)

---
# B. Klasifikasi Multikelas
Target adalah **satu kolom dengan lebih dari 2 kelas**, dan tiap baris hanya punya satu kelas. Metrik utama: accuracy dan **F1 macro** (menghargai kelas kecil), ditambah confusion matrix.

## B1. Date Fruit Datasets
- **Data:** 898 sampel dengan 34 fitur (morfologi, bentuk, warna) dari 7 jenis kurma: Barhee, Deglet Nour, Sukkary, Rotab Mozafati, Ruthana, Safawi, Sagai.
- **Identifikasi multikelas:** satu kolom `Class` dengan 7 nilai.
- Link: https://www.kaggle.com/datasets/muratkokludataset/date-fruit-datasets

In [ ]:
folder = download("muratkokludataset/date-fruit-datasets")
df = read_any(find_file(folder, patterns=["Date_Fruit"]))
print(df.shape)

target = pick_col(df, ["Class"])
y = df[target].astype(str).str.strip()
X = df.drop(columns=[target])

# --- Identifikasi ---
print("Jumlah kelas:", y.nunique(), "=> multikelas:", y.nunique() > 2)
print(y.value_counts())
plot_dist(y, "Date Fruit: distribusi kelas", rot=45)

In [ ]:
run_multiclass("Date Fruit", X, y)

## B2. Stellar Classification (SDSS17)
- **Data:** 100.000 observasi dari Sloan Digital Sky Survey dengan 17 kolom fitur dan 1 kolom `class`: GALAXY, STAR, atau QSO (kuasar).
- **Identifikasi multikelas:** 3 kelas yang tidak seimbang (GALAXY sekitar 59%, STAR sekitar 22%, QSO sekitar 19%).
- Kolom ID (`obj_ID`, `run_ID`, dan sejenisnya) dibuang karena hanya penanda, bukan sifat fisik objek.
- Link: https://www.kaggle.com/datasets/fedesoriano/stellar-classification-dataset-sdss17

In [ ]:
folder = download("fedesoriano/stellar-classification-dataset-sdss17")
df = read_any(find_file(folder, patterns=["star_classification", "stellar"]))
print(df.shape)

target = pick_col(df, ["class"])
y = df[target].astype(str).str.strip()
id_cols = [c for c in ["obj_ID", "run_ID", "rerun_ID", "cam_col", "field_ID", "spec_obj_ID", "fiber_ID"] if c in df.columns]
X = df.drop(columns=[target] + id_cols)

# --- Identifikasi ---
print("Jumlah kelas:", y.nunique(), "=> multikelas:", y.nunique() > 2)
print(y.value_counts(normalize=True).round(3))
plot_dist(y, "Stellar: distribusi kelas")

In [ ]:
run_multiclass("Stellar SDSS17", X, y)

## B3. Predict Students' Dropout and Academic Success
- **Data:** 4.424 mahasiswa dengan sekitar 36 fitur (demografi, sosial-ekonomi, nilai semester 1-2).
- **Identifikasi multikelas:** kolom `Target` dengan 3 kelas (Dropout, Enrolled, Graduate), dan satu kelas jauh lebih dominan.
- Link: https://www.kaggle.com/datasets/thedevastator/higher-education-predictors-of-student-retention

In [ ]:
folder = download("thedevastator/higher-education-predictors-of-student-retention")
path = find_file(folder, patterns=["dataset", "train", "student"])
print("File:", path.name)
df = read_any(path)
print(df.shape)

target = pick_col(df, ["Target", "target", "Class", "Status"])
y = df[target].astype(str).str.strip()
X = df.drop(columns=[target])

# --- Identifikasi ---
print("Jumlah kelas:", y.nunique(), "=> multikelas:", y.nunique() > 2)
print(y.value_counts(normalize=True).round(3))
plot_dist(y, "Student Dropout: distribusi kelas")

In [ ]:
run_multiclass("Student Dropout", X, y)

---
# C. Klasifikasi Multilabel
Satu baris bisa punya **beberapa label sekaligus**. Target diubah menjadi matriks biner (satu kolom per label). Metrik yang dipakai:
- **Hamming loss**: proporsi label yang salah (makin kecil makin baik).
- **Subset accuracy**: semua label pada satu baris harus benar (sangat ketat).
- **F1 micro / macro / samples**: F1 dengan agregasi berbeda.

## C1. Pokemon: prediksi tipe (Type 1 + Type 2)
- **Data:** 898 Pokemon dengan `Type 1` dan `Type 2` (sebagian hanya punya satu tipe).
- **Identifikasi multilabel:** tipe digabung menjadi himpunan, lalu di-binarisasi menjadi 18 label. Pokemon dual-type punya 2 label aktif.
- **Fitur:** hanya angka stat (HP, Attack, dll.). Tipe tidak bisa ditebak akurat dari stat saja, jadi skor yang rendah itu wajar.
- Link: https://www.kaggle.com/datasets/jaidalmotra/pokemon-dataset

In [ ]:
folder = download("jaidalmotra/pokemon-dataset")
df = read_any(find_file(folder))
print(df.shape)

t1 = pick_col(df, ["Type 1", "Type1", "type1"])
t2 = pick_col(df, ["Type 2", "Type2", "type2"])

def as_type_set(row):
    vals = {str(v).strip() for v in row if pd.notna(v) and str(v).strip().lower() not in ("", "nan", "none")}
    return sorted(vals)

types = df[[t1, t2]].apply(as_type_set, axis=1)
mlb = MultiLabelBinarizer()
Y = pd.DataFrame(mlb.fit_transform(types), columns=mlb.classes_, index=df.index)

skip = {"number", "#", "name", "total", t1.lower(), t2.lower()}
X = df.drop(columns=[c for c in df.columns if str(c).lower() in skip])
X = clean_features(X.select_dtypes(include=["number", "bool"]))
print("Fitur:", list(X.columns))

# --- Identifikasi ---
check_multilabel(Y, "Pokemon types")

In [ ]:
Xtr, Xte, Ytr, Yte = train_test_split(X, Y, test_size=0.2, random_state=RANDOM_STATE)

models = {
    "OvR-LogReg": Pipeline([("prep", make_preprocessor(Xtr)),
                            ("clf", OneVsRestClassifier(LogisticRegression(max_iter=2000, class_weight="balanced")))]),
    "RandomForest": Pipeline([("prep", make_preprocessor(Xtr, scale=False)),
                              ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced_subsample",
                                                             n_jobs=-1, random_state=RANDOM_STATE))]),
}
run_multilabel("Pokemon types", models, Xtr, Xte, Ytr, Yte, list(Y.columns))

## C2. NIH Chest X-ray (versi metadata)
- **Data:** 112.120 gambar rontgen dada dari 30.805 pasien. Kolom `Finding Labels` berisi 14 penyakit dipisah `|`, mis. `Cardiomegaly|Effusion`. Gambar tanpa penyakit berlabel `No Finding`.
- **Identifikasi multilabel:** satu gambar bisa punya beberapa penyakit sekaligus. Label dipecah dan di-binarisasi, dan `No Finding` menjadi baris tanpa label aktif.
- **Penting:** file gambarnya sangat besar, jadi notebook ini hanya mengunduh file **metadata** `Data_Entry_2017.csv` (usia, gender, posisi foto, dll.). Fitur metadata lemah untuk memprediksi penyakit, jadi skornya rendah. Untuk hasil yang bagus, gunakan gambarnya dengan CNN (mis. DenseNet-121).
- Split dilakukan **per pasien** (`GroupShuffleSplit`) agar gambar pasien yang sama tidak bocor ke data uji.
- Link: https://www.kaggle.com/datasets/nih-chest-xrays/data

In [ ]:
nih_csv = None
for fname in ["Data_Entry_2017.csv", "Data_Entry_2017_v2020.csv"]:
    try:
        nih_csv = download("nih-chest-xrays/data", path=fname)
        print("Berhasil mengunduh:", fname)
        break
    except Exception as e:
        print("Gagal mengunduh", fname, "->", type(e).__name__)
assert nih_csv is not None, "File metadata tidak ditemukan. Cek nama file di halaman dataset Kaggle."

df = pd.read_csv(nih_csv)
df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")]
print(df.shape)

lab_col = pick_col(df, ["Finding Labels"])
label_lists = df[lab_col].str.split("|").apply(lambda L: [x.strip() for x in L if x.strip() != "No Finding"])
mlb = MultiLabelBinarizer()
Y = pd.DataFrame(mlb.fit_transform(label_lists), columns=mlb.classes_, index=df.index)

groups = df[pick_col(df, ["Patient ID"])]
drop_cols = [c for c in [pick_col(df, ["Image Index"]), lab_col, pick_col(df, ["Patient ID"])]]
X = df.drop(columns=drop_cols)

# --- Identifikasi ---
check_multilabel(Y, "NIH Chest X-ray")
display(X.head())

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
tr_idx, te_idx = next(gss.split(X, groups=groups))
Xtr, Xte = X.iloc[tr_idx], X.iloc[te_idx]
Ytr, Yte = Y.iloc[tr_idx], Y.iloc[te_idx]

models = {
    "OvR-LogReg": Pipeline([("prep", make_preprocessor(Xtr)),
                            ("clf", OneVsRestClassifier(LogisticRegression(max_iter=1000, class_weight="balanced")))]),
    "RandomForest": Pipeline([("prep", make_preprocessor(Xtr, scale=False)),
                              ("clf", RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=20,
                                                             class_weight="balanced_subsample",
                                                             n_jobs=-1, random_state=RANDOM_STATE))]),
}
run_multilabel("NIH Chest X-ray (metadata)", models, Xtr, Xte, Ytr, Yte, list(Y.columns))

## C3. Indonesian Abusive and Hate Speech (Twitter)
- **Data:** 13.169 tweet berbahasa Indonesia dengan 12 label: `HS`, `Abusive`, `HS_Individual`, `HS_Group`, `HS_Religion`, `HS_Race`, `HS_Physical`, `HS_Gender`, `HS_Other`, `HS_Weak`, `HS_Moderate`, `HS_Strong`. Tiap label bernilai 1 jika tweet termasuk label tersebut.
- **Identifikasi multilabel:** 12 kolom biner yang bisa aktif bersamaan (mis. tweet yang `HS`, `Abusive`, dan `HS_Group` sekaligus).
- **Fitur:** teks tweet diubah ke TF-IDF (unigram + bigram).
- Link: https://www.kaggle.com/datasets/ilhamfp31/indonesian-abusive-and-hate-speech-twitter-text

In [ ]:
folder = download("ilhamfp31/indonesian-abusive-and-hate-speech-twitter-text")
data_path = [p for p in Path(folder).rglob("data.csv")]
data_path = data_path[0] if data_path else find_file(folder, patterns=["data"])
df = pd.read_csv(data_path, encoding="latin-1")
print(df.shape)

LABELS = ["HS", "Abusive", "HS_Individual", "HS_Group", "HS_Religion", "HS_Race",
          "HS_Physical", "HS_Gender", "HS_Other", "HS_Weak", "HS_Moderate", "HS_Strong"]
label_cols = [c for c in LABELS if c in df.columns]
text_col = pick_col(df, ["Tweet", "text"])

def clean_text(t):
    t = str(t).lower()
    t = t.replace("\\n", " ")
    t = re.sub(r"\b(user|url|rt)\b", " ", t)
    t = re.sub(r"[^a-z0-9\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

texts = df[text_col].map(clean_text)
Y = df[label_cols].astype(int)

# --- Identifikasi ---
check_multilabel(Y, "Hate speech Indonesia")
display(df[[text_col] + label_cols[:4]].head())

In [ ]:
Xtr, Xte, Ytr, Yte = train_test_split(texts, Y, test_size=0.2, random_state=RANDOM_STATE)

tfidf = lambda: TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True)
models = {
    "TFIDF+OvR-LogReg": Pipeline([("tfidf", tfidf()),
                                  ("clf", OneVsRestClassifier(LogisticRegression(max_iter=1000, class_weight="balanced")))]),
    "TFIDF+OvR-LinearSVC": Pipeline([("tfidf", tfidf()),
                                     ("clf", OneVsRestClassifier(LinearSVC(class_weight="balanced")))]),
}
run_multilabel("Hate speech Indonesia", models, Xtr, Xte, Ytr, Yte, label_cols)

---
# Ringkasan Hasil

In [ ]:
summary = pd.DataFrame(RESULTS)
for task, grp in summary.groupby("task", sort=False):
    print(f"\n=== {task.upper()} ===")
    display(grp.drop(columns=["task"]).dropna(axis=1, how="all").round(3).reset_index(drop=True))

## Kesimpulan & catatan

1. **Biner**: periksa kolom target punya tepat 2 nilai. Untuk data tidak seimbang (Exoplanet, Pokemon legendary), accuracy menyesatkan, jadi lihat recall, F1, dan PR-AUC.
2. **Multikelas**: satu kolom target dengan >2 kelas dan tiap baris tepat satu kelas. Gunakan F1 macro dan confusion matrix untuk melihat kelas mana yang tertukar.
3. **Multilabel**: target berupa beberapa kolom biner. Buktikan dengan *label cardinality* dan persentase baris dengan >1 label. Gunakan Hamming loss, subset accuracy, dan F1 micro/macro/samples.
4. Skor rendah pada **Pokemon types** dan **NIH (metadata)** itu wajar karena fiturnya memang tidak cukup informatif. Contoh ini tetap berguna untuk memahami mekanisme multilabel.

**Ide pengembangan:** tuning hyperparameter (GridSearchCV), threshold tuning per label, `class_weight`/SMOTE untuk data tidak seimbang, dan CNN pada gambar NIH Chest X-ray.